In [1]:
from google.colab import files
import io

uploaded = files.upload()

# Assuming only one file is uploaded
file_name = list(uploaded.keys())[0]
!unzip 'English to Bengali For Machine Translation Pre-Train.zip'


Saving English to Bengali For Machine Translation Pre-Train.zip to English to Bengali For Machine Translation Pre-Train.zip
Archive:  English to Bengali For Machine Translation Pre-Train.zip
  inflating: english_to_bangla.csv   
  inflating: EBook_of_The_Bhagavad-Gita_Bengali.txt  
  inflating: EBook_of_The_Bhagavad-Gita_English.txt  


In [2]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 8.0 MB/s eta 0:00:00


In [3]:
import torch
from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from datasets import Dataset
import pandas as pd
import numpy as np
from sacrebleu import corpus_bleu
import warnings
warnings.filterwarnings("ignore")

CONFIG = {
    "model_name": "facebook/mbart-large-50-many-to-many-mmt",
    "source_lang": "en_XX",
    "target_lang": "bn_IN",
    "data_file_path": "english_to_bangla.csv",
    "num_sentences": 10,
    "max_sentence_length": 128,
    "vocab_size": 250027,
    "d_model": 1024,
    "learning_rate": 3e-5,
    "dff": 4096,
    "batch_size": 4,
    "num_epochs": 10,
    "warmup_steps": 500,
    "early_stopping_patience": 3,
    "early_stopping_threshold": 0.01,
    "save_steps": 500,
    "eval_steps": 500,
    "logging_steps": 100,
    "gradient_accumulation_steps": 2,
    "weight_decay": 0.01,
    "adam_epsilon": 1e-8,
    "max_grad_norm": 1.0,
    "output_dir": "./mbart-en-bn-finetuned",
    "num_beams": 5,
    "repetition_penalty": 1.2,
    "length_penalty": 1.0
}

class TranslationDataset:
    def __init__(self, tokenizer, source_texts, target_texts, max_length):
        self.tokenizer = tokenizer
        self.source_texts = source_texts
        self.target_texts = target_texts
        self.max_length = max_length

    def __len__(self):
        return len(self.source_texts)

    def __getitem__(self, idx):
        source_text = str(self.source_texts[idx])
        target_text = str(self.target_texts[idx])

        source_encoding = self.tokenizer(
            source_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        target_encoding = self.tokenizer(
            target_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = target_encoding["input_ids"].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": source_encoding["input_ids"].flatten(),
            "attention_mask": source_encoding["attention_mask"].flatten(),
            "labels": labels.flatten(),
        }

def load_data_from_csv(file_path, num_sentences):
    try:
        df = pd.read_csv(file_path)
        if num_sentences > 0:
            df = df.head(num_sentences)

        english_texts = df['en'].astype(str).tolist()
        bengali_texts = df['bn'].astype(str).tolist()

        english_texts = [text[:CONFIG["max_sentence_length"]] for text in english_texts]
        bengali_texts = [text[:CONFIG["max_sentence_length"]] for text in bengali_texts]

        return english_texts, bengali_texts
    except Exception as e:
        print(f"Error loading data: {e}")
        return [], []

def setup_model_and_tokenizer():
    tokenizer = MBart50TokenizerFast.from_pretrained(CONFIG["model_name"])
    model = MBartForConditionalGeneration.from_pretrained(CONFIG["model_name"])

    tokenizer.src_lang = CONFIG["source_lang"]
    tokenizer.tgt_lang = CONFIG["target_lang"]

    return model, tokenizer

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    try:
        bleu_score = corpus_bleu(decoded_preds, [decoded_labels]).score
        return {"bleu": bleu_score}
    except:
        return {"bleu": 0.0}

def create_trainer(model, tokenizer, train_dataset, eval_dataset):
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        return_tensors="pt"
    )

    training_args = TrainingArguments(
        output_dir=CONFIG["output_dir"],
        per_device_train_batch_size=CONFIG["batch_size"],
        per_device_eval_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        num_train_epochs=CONFIG["num_epochs"],
        learning_rate=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
        adam_epsilon=CONFIG["adam_epsilon"],
        max_grad_norm=CONFIG["max_grad_norm"],
        warmup_steps=CONFIG["warmup_steps"],
        save_steps=CONFIG["save_steps"],
        eval_steps=CONFIG["eval_steps"],
        eval_strategy ="steps",
        save_strategy="steps",
        logging_steps=CONFIG["logging_steps"],
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        report_to=[],
        dataloader_pin_memory=False,
        gradient_checkpointing=True,
        fp16=True if torch.cuda.is_available() else False,
        save_total_limit=3,
        remove_unused_columns=False,
    )

    early_stopping_callback = EarlyStoppingCallback(
        early_stopping_patience=CONFIG["early_stopping_patience"],
        early_stopping_threshold=CONFIG["early_stopping_threshold"]
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[early_stopping_callback],
    )

    return trainer

def translate_text(model, tokenizer, text, device):
    model.eval()

    tokenizer.src_lang = CONFIG["source_lang"]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=CONFIG["max_sentence_length"],
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id[CONFIG["target_lang"]],
            max_length=CONFIG["max_sentence_length"],
            num_beams=CONFIG["num_beams"],
            repetition_penalty=CONFIG["repetition_penalty"],
            length_penalty=CONFIG["length_penalty"],
            early_stopping=True
        )

    translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    return translation

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    english_texts, bengali_texts = load_data_from_csv(
        CONFIG["data_file_path"],
        CONFIG["num_sentences"]
    )

    if len(english_texts) == 0:
        print("No data loaded. Please check the CSV file.")
        return

    print(f"Loaded {len(english_texts)} translation pairs")

    split_idx = int(0.9 * len(english_texts))
    train_en, train_bn = english_texts[:split_idx], bengali_texts[:split_idx]
    val_en, val_bn = english_texts[split_idx:], bengali_texts[split_idx:]

    global tokenizer
    model, tokenizer = setup_model_and_tokenizer()
    model.to(device)

    train_dataset = TranslationDataset(tokenizer, train_en, train_bn, CONFIG["max_sentence_length"])
    val_dataset = TranslationDataset(tokenizer, val_en, val_bn, CONFIG["max_sentence_length"])

    trainer = create_trainer(model, tokenizer, train_dataset, val_dataset)

    print("Starting fine-tuning...")
    trainer.train()

    trainer.save_model(CONFIG["output_dir"])
    tokenizer.save_pretrained(CONFIG["output_dir"])
    print("Model saved successfully!")

    test_sentences = [
        "a child in a pink dress is climbing up a set of stairs in an entry way .",
        "a girl going into a wooden building .",
        "a dog is running in the snow",
        "a dog running",
        "Hello, how are you?",
        "a man in an orange hat starring at something .",
        "I love you.",
        "a little girl climbing into a wooden playhouse .",
        "What is your name?",
        "two dogs of different breeds looking at each other on the road .",
        "Good morning.",
        "Thank you very much.",
        "Hello, how are you?",
        "I love you.",
        "What is your name?",
        "Good morning.",
        "Thank you very much.",
        "The weather is nice today."
    ]

    print("\nTranslation Results:")
    print("=" * 80)

    for i, sentence in enumerate(test_sentences, 1):
        try:
            translation = translate_text(model, tokenizer, sentence, device)
            print(f"{i:2d}. English: {sentence}")
            print(f"    Bengali: {translation}")
            print("-" * 80)
        except Exception as e:
            print(f"{i:2d}. Error translating: {sentence}")
            print(f"    Error: {e}")
            print("-" * 80)


In [4]:
main()

Using device: cuda
Loaded 10 translation pairs


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Starting fine-tuning...


Step,Training Loss,Validation Loss


Model saved successfully!

Translation Results:
 1. English: a child in a pink dress is climbing up a set of stairs in an entry way .
    Bengali: একটি রোজ ্ টিক জামাকাপ নিয়ে শিশু একটা বেড়ার মধে ্ য দিয়ে উঠেছে
--------------------------------------------------------------------------------
 2. English: a girl going into a wooden building .
    Bengali: একটি মেয়ে একটি কৃতির বাড়িতে যাচ ্ ছে ।
--------------------------------------------------------------------------------
 3. English: a dog is running in the snow
    Bengali: কুকুর একটা স ্ নোতে দৌড়াচ ্ ছ
--------------------------------------------------------------------------------
 4. English: a dog running
    Bengali: একটি কুকুরকে নাড়ে বেড়ানো
--------------------------------------------------------------------------------
 5. English: Hello, how are you?
    Bengali: হাইলো, কেমন আছেন আপনারা?
--------------------------------------------------------------------------------
 6. English: a man in an orange hat starring at somet